# Training Effectiveness Analysis

## Project Overview
Analyze employee training participation and post-training outcomes to assess program effectiveness across departments, roles, and training types.

## Learning Objectives
- Measure training participation rates and completion patterns
- Analyze pre/post training performance changes
- Compare effectiveness across training types and delivery formats
- Identify which programs deliver the highest ROI on employee performance

## Problem Statement
L&D leadership needs to understand which training programs actually improve employee performance and which are underperforming. They also need to optimize training budget allocation based on measurable outcomes.

## Why This Project Matters
Organizations spend billions on training annually, but most lack rigorous measurement of training effectiveness. Data-driven training evaluation ensures budget is spent on programs that genuinely develop talent.

## Dataset Overview
Synthetic training dataset: ~4,000 training records with pre/post performance scores, completion data, and employee context.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 4000
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'Finance', 'Operations', 'Support'],
                                 n, p=[0.22, 0.20, 0.15, 0.13, 0.17, 0.13])
levels = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead'], n, p=[0.28, 0.32, 0.25, 0.15])
training_type = np.random.choice(['Technical Skills', 'Leadership', 'Compliance', 'Soft Skills', 'Product Knowledge', 'Onboarding'],
                                   n, p=[0.25, 0.15, 0.18, 0.14, 0.16, 0.12])
delivery = np.random.choice(['Online Self-Paced', 'Live Virtual', 'In-Person', 'Blended'], n, p=[0.35, 0.25, 0.20, 0.20])
duration_hours = np.clip(np.random.lognormal(2.0, 0.7, n), 1, 80).round(1)

pre_score = np.clip(np.random.normal(60, 15, n), 10, 95).round(1)
# Post-score depends on training type and completion
base_improvement = {'Technical Skills': 12, 'Leadership': 8, 'Compliance': 5,
                     'Soft Skills': 7, 'Product Knowledge': 10, 'Onboarding': 15}
improvement = np.array([base_improvement[t] for t in training_type])
completed = np.random.random(n) < np.where(delivery == 'Online Self-Paced', 0.70, 0.88)
post_score = np.where(completed,
    np.clip(pre_score + improvement + np.random.normal(0, 6, n), 15, 100),
    np.clip(pre_score + np.random.normal(1, 3, n), 10, 100)
).round(1)

satisfaction = np.where(completed, np.clip(np.random.normal(3.8, 0.7, n), 1, 5), np.nan).round(1)
cost_per_head = np.where(delivery == 'In-Person', np.random.uniform(200, 800, n),
                 np.where(delivery == 'Blended', np.random.uniform(150, 500, n),
                 np.where(delivery == 'Live Virtual', np.random.uniform(50, 200, n),
                          np.random.uniform(20, 100, n)))).round(0)

df = pd.DataFrame({
    'RecordID': [f'TR{i:05d}' for i in range(n)],
    'Department': departments, 'Level': levels,
    'TrainingType': training_type, 'DeliveryFormat': delivery,
    'DurationHours': duration_hours,
    'PreScore': pre_score, 'PostScore': post_score,
    'Completed': completed, 'SatisfactionScore': satisfaction,
    'CostPerHead': cost_per_head,
})
df['ScoreChange'] = (df['PostScore'] - df['PreScore']).round(1)
df['Improved'] = (df['ScoreChange'] > 0).astype(int)

print(f'Dataset: {df.shape}')
print(f'Completion rate: {df["Completed"].mean():.1%}')
print(f'Avg score change: {df["ScoreChange"].mean():.1f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values per column:')
print(df.isnull().sum())
print(f'\nTraining type distribution:\n{df["TrainingType"].value_counts()}')
print(f'\nDelivery format distribution:\n{df["DeliveryFormat"].value_counts()}')

## Overall Training Effectiveness

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['ScoreChange'], bins=30, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Score Change Distribution')
axes[0].set_xlabel('Post - Pre Score')

completed_df = df[df['Completed']]
axes[1].scatter(completed_df['PreScore'], completed_df['PostScore'], alpha=0.15, s=8, color='steelblue')
axes[1].plot([10, 100], [10, 100], 'r--', alpha=0.5, label='No Change')
axes[1].set_xlabel('Pre-Score')
axes[1].set_ylabel('Post-Score')
axes[1].set_title('Pre vs Post Score (Completed)')
axes[1].legend()

axes[2].hist(completed_df['SatisfactionScore'].dropna(), bins=20, edgecolor='black', color='teal', alpha=0.7)
axes[2].set_title('Satisfaction Score Distribution')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'overall_effectiveness.png'), dpi=100, bbox_inches='tight')
plt.show()

## Effectiveness by Training Type

In [ ]:
type_eff = df[df['Completed']].groupby('TrainingType').agg(
    records=('RecordID', 'count'),
    avg_change=('ScoreChange', 'mean'),
    pct_improved=('Improved', 'mean'),
    avg_satisfaction=('SatisfactionScore', 'mean'),
    avg_cost=('CostPerHead', 'mean'),
).round(2)
type_eff = type_eff.sort_values('avg_change', ascending=False)
print('Effectiveness by Training Type (completed only):')
print(type_eff)

fig, ax = plt.subplots(figsize=(10, 5))
type_eff['avg_change'].plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Avg Score Change by Training Type')
ax.set_xlabel('Avg Score Change')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'type_effectiveness.png'), dpi=100, bbox_inches='tight')
plt.show()

## Completion Rates by Delivery Format

In [ ]:
delivery_stats = df.groupby('DeliveryFormat').agg(
    records=('RecordID', 'count'),
    completion_rate=('Completed', 'mean'),
    avg_change_completed=('ScoreChange', lambda x: x[df.loc[x.index, 'Completed']].mean() if df.loc[x.index, 'Completed'].any() else 0),
    avg_cost=('CostPerHead', 'mean'),
).round(3)
print('Stats by Delivery Format:')
print(delivery_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
delivery_stats['completion_rate'].plot.bar(ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Completion Rate by Delivery')
axes[0].tick_params(axis='x', rotation=0)

df_comp = df[df['Completed']]
sns.boxplot(data=df_comp, x='DeliveryFormat', y='ScoreChange', ax=axes[1])
axes[1].set_title('Score Change by Delivery (Completed)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'delivery_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Cost-Effectiveness Analysis

In [ ]:
cost_eff = df[df['Completed']].groupby('TrainingType').agg(
    avg_change=('ScoreChange', 'mean'),
    avg_cost=('CostPerHead', 'mean'),
).round(2)
cost_eff['cost_per_point'] = (cost_eff['avg_cost'] / cost_eff['avg_change'].clip(lower=0.1)).round(2)
cost_eff = cost_eff.sort_values('cost_per_point')
print('Cost per Performance Point by Training Type:')
print(cost_eff)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(cost_eff['avg_cost'], cost_eff['avg_change'], s=150, edgecolor='black', color='coral', zorder=5)
for idx, row in cost_eff.iterrows():
    ax.annotate(idx, (row['avg_cost'], row['avg_change']), textcoords='offset points', xytext=(8, 5), fontsize=9)
ax.set_xlabel('Avg Cost per Head ($)')
ax.set_ylabel('Avg Score Improvement')
ax.set_title('Training Cost vs Effectiveness')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cost_effectiveness.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department Training Patterns

In [ ]:
dept_train = df.groupby('Department').agg(
    total_records=('RecordID', 'count'),
    completion_rate=('Completed', 'mean'),
    avg_change=('ScoreChange', 'mean'),
    total_spend=('CostPerHead', 'sum'),
).round(2)
dept_train['spend_per_record'] = (dept_train['total_spend'] / dept_train['total_records']).round(0)
print('Training by Department:')
print(dept_train.sort_values('avg_change', ascending=False))

## Duration vs Effectiveness

In [ ]:
df_comp = df[df['Completed']].copy()
df_comp['DurationBand'] = pd.cut(df_comp['DurationHours'], bins=[0, 4, 8, 16, 40, 999],
                                   labels=['<4h', '4-8h', '8-16h', '16-40h', '40h+'])
dur_eff = df_comp.groupby('DurationBand').agg(
    records=('RecordID', 'count'),
    avg_change=('ScoreChange', 'mean'),
    avg_satisfaction=('SatisfactionScore', 'mean'),
).round(2)
print('Effectiveness by Duration:')
print(dur_eff)

fig, ax = plt.subplots(figsize=(8, 5))
dur_eff['avg_change'].plot.bar(ax=ax, edgecolor='black', color='teal')
ax.set_title('Avg Score Change by Training Duration')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'duration_effectiveness.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Onboarding and Technical Skills** training show the largest score improvements — these are knowledge-gap closure programs with clear measurable outcomes
- **Online self-paced** has notably lower completion rates (~70% vs ~88% for instructor-led) — engagement interventions are needed
- **Cost-effectiveness varies** significantly — compliance training costs more per performance point gained
- **In-person delivery** costs 3-4x more but doesn't proportionally improve outcomes — blended approaches may offer the best balance
- **Longer programs** don't always produce better outcomes — diminishing returns often appear past 16 hours
- **Satisfaction ≠ effectiveness** — highly rated programs don't always produce the largest score gains

## Limitations
- Synthetic data with pre-programmed improvement rates
- Pre/post score is a simplified effectiveness measure
- No control group — can't attribute improvement solely to training
- No long-term retention measurement (30/60/90-day follow-up)
- Self-reported satisfaction may not reflect actual learning

## How to Improve This Project
- Add control groups for causal attribution
- Include 30/60/90-day knowledge retention assessments
- Track on-the-job behavior change post-training
- Add manager assessment of skill application
- Build a training recommendation engine based on skill gaps

## Production Considerations
- L&D dashboard with real-time training effectiveness metrics
- Automated low-completion alerts for self-paced programs
- Training ROI reports for budget planning
- Personalized training recommendations based on skill gaps and role requirements

## Common Mistakes
- Measuring only satisfaction (Level 1 Kirkpatrick) and calling it effectiveness
- Not comparing completers vs non-completers
- Treating all training types equally in budget allocation
- Ignoring selection bias (high performers may self-select into training)

## Mini Challenge / Exercises
1. Calculate the overall training ROI assuming each performance point gained is worth $500 in productivity.
2. Which TrainingType × DeliveryFormat combination has the best cost-per-point ratio?
3. If you cut all programs with <5 avg score change, how much budget would you save?
4. Build a simple training prioritization ranking using a composite of effectiveness, cost, and completion rate.

## Final Summary / Key Takeaways
- Training effectiveness measurement goes beyond satisfaction surveys
- Pre/post score comparison reveals which programs actually build skills
- Cost-per-point analysis enables data-driven budget allocation
- Completion rate differences by delivery format highlight engagement challenges
- Organizations should prioritize high-ROI programs and redesign low-effectiveness ones